# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR^2 mass spectrometry dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant) library.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install the mlcroissant library
!pip install mlcroissant

## 1. Data Loading
Load and inspect dataset metadata from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Citation: {getattr(dataset.metadata, 'citeAs', None)}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")
print(f"Identifier: {getattr(dataset.metadata, 'identifier', None)}")

## 2. Data Overview
Explore available record sets, their `@id`s, and field metadata.

In [ ]:
# List all record sets, their @id, and their fields
record_sets = dataset.list_record_sets()
print(f"Found {len(record_sets)} record sets in the dataset.\n")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}  |  name: {rs.get('name', None)}  |  description: {rs.get('description', '')}")
    fields = dataset.list_fields(record_set=rs['@id'])
    print("    Fields and @ids:")
    for f in fields:
        print(f"      - {f['@id']:40}  | name: {f.get('name', '')}")
    print()
if len(record_sets) == 0:
    print("No record sets found. Please check the dataset schema for record sets.")

## 3. Data Extraction
Load data from selected record set(s) into DataFrames for data analysis.
Use record set and field `@id`s as referenced in the overview above.

In [ ]:
# --- User editable: List @id's of record sets you'd like to extract ---
# Fill these in from the record set @id's shown above.
selected_record_sets = []

# Example: If you see a main record set with @id 'rs:Main', use:
# selected_record_sets = ['rs:Main']
# For this dataset, substitute with the actual @ids found in previous section.

# If none, code will raise.
if not selected_record_sets:
    print("Please set selected_record_sets to the list of record set @id(s) from previous cell (see their '@id').")
else:
    # Load all selected record sets as DataFrames
    dataframes = {}
    for rs_id in selected_record_sets:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"First few rows of Record Set '{rs_id}':")
        display(df.head())  # for Jupyter display
        print(f"Columns: {list(df.columns)}\n")

## 4. Exploratory Data Analysis (EDA)
Now you can explore, filter, and transform the records loaded into DataFrames. The exact field `@id`s to use depend on the record set and field overview above.

Common EDA tasks: filter by a numeric field, normalize it, group and aggregate by a categorical field.

In [ ]:
if not selected_record_sets:
    print("Please run and fill in the previous cell with at least one record set @id to continue.")
else:
    # --- User editable: Choose the record set @id to analyze and field @ids ---
    record_set_id = selected_record_sets[0]
    df = dataframes[record_set_id]

    # Inspect the columns to pick a numeric field (e.g., 'cr:coverage_percent', etc)
    print('Columns in DataFrame:', df.columns.tolist())

    # Set these to the field @id(s) you want to analyze:
    numeric_field_id = None  # e.g., 'cr:coverage_percent'
    group_field_id = None    # e.g., 'cr:sample_type' if present

    # Example: Fill these in as per your dataset's schema and printout
    # numeric_field_id = 'cr:MW'  # Example field for molecular weight
    # group_field_id = 'cr:sample_label'

    # EDA steps below require field ids. If not set, skip analysis.
    if numeric_field_id is None or numeric_field_id not in df.columns:
        print("Please inspect columns and set 'numeric_field_id' to a valid numeric field @id in the DataFrame.")
    else:
        # Filtering (this is just an example; edit as appropriate)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows\n")
        display(filtered_df.head())

        # Normalization
        field_norm = f"{numeric_field_id}_normalized"
        filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized field '{numeric_field_id}':")
        display(filtered_df[[numeric_field_id, field_norm]].head())

        # Group by another field if present
        if group_field_id and group_field_id in df.columns:
            grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped by '{group_field_id}':")
            display(grouped.head())

## 5. Visualization
Plot distribution(s) for selected numeric field(s) and explore relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not selected_record_sets:
    print("Load a DataFrame as above before plotting.")
elif numeric_field_id is None or numeric_field_id not in df.columns:
    print("Set 'numeric_field_id' to a numeric column in your DataFrame to plot.")
else:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Optional: Boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load and analyze a FAIR^2 dataset defined by the Croissant metadata schema.

- Inspected dataset metadata, record sets, and field structure via their `@id`s
- Loaded data using `mlcroissant` and explored it with common EDA operations
- Visualized numeric fields and grouped data as an example for deeper domain analysis

**Next steps:**
- Identify and select more specific record sets and fields of interest by their `@id`
- Apply domain-specific cleaning, normalization, and statistical analysis tailored to proteomics studies
- Integrate findings with biological context or connect to external resources (such as UniProt annotations)